# CAD editing: verifier-guided distillation, Astra teacher -> Qwen2.5-Coder-3B student

Same QLoRA recipe, splits and scorer as the completed 3B run; only the training targets change.
Targets are `PLAN:` (quoted zero-based original lines) then `PATCH:` (the JSON patch).

* **template**: plans generated from the reference patch, no teacher (control; isolates the plan format).
* **distill**: BenchCAD training rows whose Astra candidates pass the target-free CAD verifier (execution gates + numeric checks + geometric consensus of n samples) use the teacher's answer.
* **distill-ref**: as distill, keeping only teacher answers that also match the training reference.

Use **Runtime -> Change runtime type -> GPU** (A100/L4 is ~4x faster than T4), then **Run all**. No API keys or passwords are in this notebook; teacher outputs were generated beforehand on the training split only.

In [1]:
!pip -q install "transformers==4.51.3" "peft==0.15.2" accelerate bitsandbytes "cadquery==2.8.0"
!apt-get -qq update && apt-get -qq install -y libglu1-mesa libgl1
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime first'
print(torch.cuda.get_device_name(0), torch.cuda.mem_get_info())

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
NVIDIA A100-SXM4-80GB (84083212288, 85094825984)


In [2]:
#@title Unpack data, scripts and test reference STEP files
import base64, hashlib, io, tarfile
from pathlib import Path
encoded = """
...
"""
bundle = base64.b64decode(encoded)
assert hashlib.sha256(bundle).hexdigest() == "b8b9b4f7b4a440632a991f7979d1e9dd4e99de2a899ed0c8f55f6006d8486d2d"
ROOT = Path('/content/cad_distill'); ROOT.mkdir(exist_ok=True)
tarfile.open(fileobj=io.BytesIO(bundle), mode='r:gz').extractall(ROOT)
for v in ['template', 'distill']:
    import json; m = json.loads((ROOT/'data'/v/'manifest.json').read_text())
    print(v, m['counts']['train'], m.get('acceptance'), m.get('teacher_audit'))

template {'template': 2540, 'teacher_verified': 0} None None
distill {'template': 2099, 'teacher_verified': 441} verifier gates + consensus>=2 {'tasks': 468, 'accepted': 441, 'accepted_reference_strict': 364}


/tmp/ipykernel_76583/1068288198.py:4752: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tarfile.open(fileobj=io.BytesIO(bundle), mode='r:gz').extractall(ROOT)


In [3]:
#@title Which student to train
VARIANTS = ['distill', 'distill-ref']  #@param {type:"raw"}
# distill: verifier-selected teacher answers (consensus of 2 Astra samples), no reference used
# distill-ref: additionally keeps only teacher answers whose solid matches the reference (training split)
# Each takes ~1.3 h on an L4; drop one to halve the run. 'template' results already exist.

## Train, evaluate and decode validation (background job)

Two epochs; final checkpoint; greedy decoding on all 252 held-out tasks, then on the 172 validation tasks (used offline to check the plan-quote anchoring rule on data that played no part in designing it). Runs as a background process logging to `train.log`, so a dropped VS Code or browser connection does not stop it: reconnect and re-run the wait cell. A completed variant is never retrained; an incomplete one is moved aside and restarted.

In [4]:
import json, os, subprocess, sys, time
LOG, PID = ROOT/'train.log', ROOT/'train.pid'
def status(v):
    m = ROOT/f'results-{v}'/'run_manifest.json'
    return json.loads(m.read_text())['status'] if m.exists() else None
def alive():
    if LOG.exists() and 'EXIT=' in LOG.read_text(errors='replace')[-300:]: return False
    try:
        os.kill(int(PID.read_text()), 0); return True
    except (OSError, ValueError):
        return False
steps = []
for v in VARIANTS:
    run = ROOT/f'results-{v}'
    if status(v) != 'completed':
        if run.exists() and not alive(): run.rename(run.with_name(f'{run.name}.incomplete-{int(time.time())}'))
        steps.append(f'{sys.executable} -u {ROOT}/scripts/train_distill_cad_colab.py --data {ROOT}/data/{v} --output {run}')
    if not (run/'validation-predictions.jsonl').exists():
        steps.append(f'{sys.executable} -u {ROOT}/scripts/decode_split_colab.py --data {ROOT}/data/{v} --run {run} --split validation')
if alive():
    print('Job already running: go to the next cell')
elif not steps:
    print('All requested variants are trained and decoded')
else:
    proc = subprocess.Popen(f'(cd {ROOT}/scripts && ' + ' && '.join(steps) + f') > {LOG} 2>&1; echo EXIT=$? >> {LOG}', shell=True, start_new_session=True)
    PID.write_text(str(proc.pid)); print('Started', len(steps), 'steps, pid', proc.pid)

Started 2 steps, pid 77164


In [5]:
#@title Wait for training (re-run after a reconnect)
seen = 0
while True:
    lines = LOG.read_text(errors='replace').replace('\r', '\n').splitlines() if LOG.exists() else []
    for line in lines[seen:]:
        if any(k in line for k in ("'loss'", "'eval_loss'", 'trained ', 'validation ', 'EXIT=', 'Error', 'Traceback', 'train_runtime')):
            print(line[:300], flush=True)
    seen = len(lines)
    if any(l.startswith('EXIT=') for l in lines): break
    time.sleep(60)
assert 'EXIT=0' in lines[-1], '\n'.join(lines[-60:])
print('Training, test and validation decoding finished')

{'loss': 0.3571, 'grad_norm': 1.2637543678283691, 'learning_rate': 0.0001, 'epoch': 0.04}
{'loss': 0.2975, 'grad_norm': 1.2354609966278076, 'learning_rate': 0.0001, 'epoch': 0.08}
{'loss': 0.2557, 'grad_norm': 0.9940333962440491, 'learning_rate': 0.0001, 'epoch': 0.12}
{'loss': 0.2571, 'grad_norm': 1.1698951721191406, 'learning_rate': 0.0001, 'epoch': 0.16}
{'loss': 0.2334, 'grad_norm': 0.9922265410423279, 'learning_rate': 0.0001, 'epoch': 0.2}
{'loss': 0.2192, 'grad_norm': 1.6923474073410034, 'learning_rate': 0.0001, 'epoch': 0.24}
{'loss': 0.2017, 'grad_norm': 1.7071043252944946, 'learning_rate': 0.0001, 'epoch': 0.28}
{'loss': 0.1959, 'grad_norm': 1.0221673250198364, 'learning_rate': 0.0001, 'epoch': 0.31}
{'loss': 0.2201, 'grad_norm': 2.236987829208374, 'learning_rate': 0.0001, 'epoch': 0.35}
{'loss': 0.2029, 'grad_norm': 1.945921540260315, 'learning_rate': 0.0001, 'epoch': 0.39}
{'loss': 0.2101, 'grad_norm': 1.2100541591644287, 'learning_rate': 0.0001, 'epoch': 0.43}
{'loss': 0.20

## Execute predictions against the reference solids

BenchCAD uses author-released STEP targets; CAD-Editor uses locally reconstructed sequence targets. Strict = volume IoU >= 0.99999.

In [6]:
for v in VARIANTS:
    run = ROOT/f'results-{v}'
    subprocess.run([sys.executable, '-u', str(ROOT/'scripts/score_multisource_cad_geometry.py'), str(run/'trained-predictions.jsonl')], check=True, cwd=ROOT/'scripts', stdout=subprocess.DEVNULL)
baselines = {"3B SFT, patch only (runs/multisource-cad-colab-20261002, local re-score)": [39, 25, 14], "Astra gpt-6-astra, zero-shot (runs/multisource-cad-astra-20261004)": [106, 89, 17]}
print(f"{'system':78s} strict  BenchCAD  CAD-Editor")
for name, (a, b, c) in baselines.items(): print(f'{name:78s} {a:6d} {b:9d} {c:11d}')
for v in VARIANTS:
    g = json.loads((ROOT/f'results-{v}'/'trained-predictions-geometry.json').read_text())
    print(f"{'3B student, '+v:78s} {g['match_strict']:6d} {g['sources']['BenchCAD']['match_strict']:9d} {g['sources']['CAD-Editor']['match_strict']:11d}")

system                                                                         strict  BenchCAD  CAD-Editor
3B SFT, patch only (runs/multisource-cad-colab-20261002, local re-score)           39        25          14
Astra gpt-6-astra, zero-shot (runs/multisource-cad-astra-20261004)                106        89          17
3B student, distill                                                                42        26          16
3B student, distill-ref                                                            33        21          12


## Save results into this notebook

`files.download` and `drive.mount` are unavailable when a local notebook runs on a Colab kernel from VS Code, so the small result files (predictions, geometry scores, metrics, manifests, training history) are printed below as base64 between BEGIN/END markers. **Save the notebook (Cmd/Ctrl+S)** so the output is written to the .ipynb. Adapter weights (~40 MB) are excluded; their SHA-256 is in run_manifest.json. In browser Colab a full ZIP is also downloaded.

In [7]:
import base64, hashlib, io, tarfile, zipfile
for v in VARIANTS:
    run = ROOT/f'results-{v}'
    buf = io.BytesIO()
    with tarfile.open(fileobj=buf, mode='w:gz') as tar:
        for f in sorted(run.rglob('*')):
            if f.is_file() and 'checkpoints' not in f.parts and (f.parent.name != 'adapter' or f.name == 'adapter_config.json'):
                tar.add(f, arcname=str(f.relative_to(run)))
    data = buf.getvalue(); text = base64.b64encode(data).decode()
    print(f'BEGIN-RESULTS {v} {len(data)} {hashlib.sha256(data).hexdigest()}')
    for i in range(0, len(text), 1000): print(text[i:i + 1000])
    print(f'END-RESULTS {v}')
    try:
        from google.colab import files
        archive = Path(f'/content/cad-distill-{v}-results.zip')
        with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
            for f in run.rglob('*'):
                if f.is_file() and 'checkpoints' not in f.parts: z.write(f, f.relative_to(run))
        files.download(str(archive))
    except Exception as error:
        print('ZIP download unavailable here:', type(error).__name__)

BEGIN-RESULTS distill 94325 ...
END-RESULTS distill


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

BEGIN-RESULTS distill-ref 97874 ...
END-RESULTS distill-ref


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>